In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config_loader as config_loader
import util.read_price as read_price
import src.ip.optimizations as optimizations

print("config_loader:", config_loader.__file__)
print("read_price:", read_price.__file__)
print("rolling_ce:", optimizations.__file__)



battery, markets = config_loader.load_config(PROJECT_ROOT / "configs" / "battery_config.yaml")
ip_cfg = markets["ip"]

ip_prob_lear = read_price.read_ip_forecast(model="LEAR", kind="probabilistic", freq="15min")
ip_prob_xgb  = read_price.read_ip_forecast(model="XGB",  kind="probabilistic", freq="15min")
ip_prob_qr   = read_price.read_ip_forecast(model="QR_old",   kind="probabilistic", freq="15min")
ip_real = read_price.read_ip_real_prices(freq="15min", keep_extra_columns=False)["Price"]

start_date = "2023-01-01"
end_date = "2023-01-06"




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
config_loader: u:\MyOwn\DA_Optimization\src\config_loader.py
read_price: u:\MyOwn\DA_Optimization\util\read_price.py
rolling_ce: u:\MyOwn\DA_Optimization\src\ip\optimizations.py


In [3]:
terminal_penalty=0.05


res = optimizations.run_ip_rolling_prob_models(
    battery=battery,
    market=ip_cfg,
    forecasts={"qr": ip_prob_qr}, # "xgb": ip_prob_xgb, "lear": ip_prob_lear, 
    real_price_series=ip_real,
    risks=[
        #{"alpha": 0.9, "lambda_cvar": 0.0},  # RN
        {"alpha": 0.9, "lambda_cvar": 0.1},  # RA
        #{"alpha": 0.9, "lambda_cvar": 2.0},  # RA
    ],
    start=start_date,
    end=end_date,
    terminal_target_kwh=battery.energy_kwh * 0.5,
    terminal_penalty=terminal_penalty,
    terminal_penalty_mode="L1",
    scenario_method="quantile_paths",
    save=True,
    tag="v1",
    cycle_penalty_eur_per_mwh = 0
)

df = res.history


Solving prob CVaR(a=0.9,lam=0.1):  99.8 %

In [4]:
terminal_penalty=0.05

res = optimizations.run_ip_rolling_prob_models(
    battery=battery,
    market=ip_cfg,
    forecasts={"qr": ip_prob_qr}, # "xgb": ip_prob_xgb, "lear": ip_prob_lear, 
    real_price_series=ip_real,
    risks=[
        #{"alpha": 0.9, "lambda_cvar": 0.0},  # RN
        {"alpha": 0.9, "lambda_cvar": 0.1},  # RA
        #{"alpha": 0.9, "lambda_cvar": 2.0},  # RA
    ],
    start=start_date,
    end=end_date,
    terminal_target_kwh=battery.energy_kwh * 0.5,
    terminal_penalty=terminal_penalty,
    terminal_penalty_mode="L1",
    save=True,
    tag="v1",
    scenario_method="copula",
    n_scenarios=100,
    lam_corr=0.995,
    lookback_days=30,
    cycle_penalty_eur_per_mwh = 0
)
df1 = res.history

Solving prob CVaR(a=0.9,lam=0.1):  99.8 %